# Customer Segmentation & Customer Value Analysis
### RFM Segmentation and Revenue-Based Customer Value on the Online Retail II Dataset

**Business question:** Which customer segments are most valuable, and where should retention efforts be prioritized?

**Approach:** Clean transaction data → calculate RFM metrics → assign behavioral segments → analyze historical revenue and annualized revenue run-rate → translate findings into retention priorities.

**Dataset:** Online Retail II — transaction data from a UK-based online retailer covering December 2009 to December 2011.

**Note:** Monetary values are reported in **GBP (£)**, consistent with the source dataset. The customer value section is descriptive and does not constitute a predictive CLV model.


## 1. Data Loading & Cleaning

Prepare a customer-level transaction dataset for RFM and value analysis. The cleaning decisions below are made explicitly because cancellations, missing customer identifiers, and invalid transaction values can distort customer-level metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [ ]:
df1 = pd.read_excel('data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('data/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df1, df2], ignore_index=True)


In [ ]:
df = df.rename(columns={
    'Customer ID': 'CustomerID',
    'Invoice': 'InvoiceNo',
    'Price': 'UnitPrice'
})


In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df['InvoiceDate'].min():%d %b %Y} to {df['InvoiceDate'].max():%d %b %Y}")


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe()


In [ ]:
# Inspect the first few rows to confirm the schema and transaction structure.
df.head(10)


In [ ]:
# Inspect the most extreme quantity to determine whether it is a genuine transaction or a cancellation.
df[df['Quantity'].abs() == 80995]


In [ ]:
# Negative prices are invalid for completed sales and are investigated before filtering.
df[df['UnitPrice'] < 0]


### Missing customer identifiers

Transactions without a CustomerID cannot be attributed to an individual customer, so they are excluded from customer-level segmentation.

In [ ]:
missing_pct = df['CustomerID'].isnull().mean() * 100
print(f"{missing_pct:.1f}% of rows have no CustomerID")

df = df.dropna(subset=['CustomerID'])


In [ ]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)


In [ ]:
cancelled = df[df['InvoiceNo'].str.startswith('C')]
cancelled_pct = len(cancelled) / len(df) * 100
print(f"{cancelled_pct:.2f}% of remaining transactions are cancellations")

df = df[~df['InvoiceNo'].str.startswith('C')]


In [ ]:
# Confirm that cancellation records are negative-quantity transactions.
(cancelled['Quantity'] < 0).mean() * 100


### Cancellations

Cancellation records represent returns/order corrections rather than completed purchases, so they are excluded from revenue and RFM calculations. The cancellation rate is retained as a useful operational signal rather than being silently discarded.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)

dupe_count = df.duplicated().sum()
print(f"{dupe_count:,} duplicate rows found ({dupe_count / len(df) * 100:.2f}%)")

df = df.drop_duplicates()


In [ ]:
print(f"Rows with non-positive UnitPrice: {(df['UnitPrice'] <= 0).sum():,}")
print(f"Rows with non-positive Quantity: {(df['Quantity'] <= 0).sum():,}")


In [ ]:
df = df[(df['UnitPrice'] > 0) & (df['Quantity'] > 0)]


In [ ]:
df.to_pickle('data/cleaned_online_retail.pkl')


In [ ]:
# Reload the cleaned dataset so the downstream analysis can be run independently from the raw-data cleaning step.
df = pd.read_pickle('data/cleaned_online_retail.pkl')


## 2. RFM Feature Engineering

Calculate Recency, Frequency, and Monetary value at customer level. Each metric is converted into a 1–5 score, where higher scores represent stronger customer behavior.

### 2.1 Calculate Recency, Frequency, and Monetary Value


In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Snapshot date = one day after the final transaction in the dataset.
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalPrice': 'sum'                                      # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']


In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_Total'] = rfm[['R_score', 'F_score', 'M_score']].astype(int).sum(axis=1)


### 2.2 Scoring Customers (1–5 scale per dimension)


In [ ]:
def segment_customer(row):
    r = row['R_score']
    f = row['F_score']
    
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r == 3 and f <= 2:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 4:
        return 'At Risk'
    elif r <= 2 and f == 3:
        return 'Needs Attention'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

### 2.3 Assigning Customers to Behavioral Segments


In [ ]:
rfm['Segment'].value_counts()

## 3. Segmentation Visuals

Compare segment size, revenue contribution, and customer behavior to understand where value is concentrated and how the segments differ.

### 3.1 Customer Count by Segment


In [ ]:
plt.figure(figsize=(10, 6))
segment_counts = rfm['Segment'].value_counts()
sns.barplot(x=segment_counts.values, y=segment_counts.index,
            hue=segment_counts.index, legend=False, palette='viridis')
plt.title('Customer Count by Segment')
plt.xlabel('Number of Customers')
plt.ylabel('Segment')
plt.tight_layout()
plt.show()


### 3.2 Revenue Contribution by Segment

In [ ]:
segment_revenue = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=segment_revenue.values, y=segment_revenue.index,
            hue=segment_revenue.index, legend=False, palette='magma')
plt.title('Total Revenue by Segment')
plt.xlabel('Total Revenue (£)')
plt.ylabel('Segment')
plt.tight_layout()
plt.show()


In [ ]:
total_revenue = rfm['Monetary'].sum()
segment_pct = (segment_revenue / total_revenue * 100).round(1)
print(segment_pct)


### 3.3 Recency vs. Frequency by Segment

In [ ]:
plt.figure(figsize=(10,7))
sns.scatterplot(data=rfm, x='Recency', y='Frequency', hue='Segment', palette='tab10', alpha=0.6)
plt.title('Customer Segments: Recency vs Frequency')
plt.tight_layout()
plt.show()

### 3.4 Recency vs. Monetary by Segment (Log Scale)

In [ ]:
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Segment', palette='tab10', alpha=0.6)
plt.yscale('log')  # Monetary is usually heavily skewed; log scale makes patterns visible
plt.title('Customer Segments: Recency vs Monetary (log scale)')
plt.tight_layout()
plt.show()

## 4. Customer Value Analysis

This section measures customer value using **historical revenue** and an **annualized revenue run-rate**. The annualized metric answers: *if a customer's observed purchasing rate continued for the next 12 months, what revenue would that imply?*

This is a descriptive/annualized revenue measure, **not a predictive CLV model**.

### 4.1 Average Order Value & Customer Lifespan


In [ ]:
customer_value = df.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

customer_value.columns = ['CustomerID', 'Frequency', 'TotalRevenue']
customer_value['AvgOrderValue'] = customer_value['TotalRevenue'] / customer_value['Frequency']


In [ ]:
lifespan = df.groupby('CustomerID')['InvoiceDate'].agg(['min', 'max']).reset_index()
lifespan['Lifespan_Days'] = (lifespan['max'] - lifespan['min']).dt.days
lifespan['Lifespan_Months'] = lifespan['Lifespan_Days'] / 30

customer_value = customer_value.merge(
    lifespan[['CustomerID', 'Lifespan_Months']],
    on='CustomerID'
)


### 4.2 Historical Revenue & Annualized Revenue Run-Rate


In [ ]:
# One-time buyers have a zero observed lifespan, so use the average positive lifespan
# only when calculating a monthly purchase-rate estimate.
avg_lifespan = customer_value.loc[
    customer_value['Lifespan_Months'] > 0, 'Lifespan_Months'
].mean()

customer_value['Lifespan_Months_Adj'] = customer_value['Lifespan_Months'].replace(0, avg_lifespan)
customer_value['PurchaseFreq_Monthly'] = customer_value['Frequency'] / customer_value['Lifespan_Months_Adj']

# Annualized revenue run-rate: implied revenue over 12 months at the observed monthly rate.
customer_value['Monthly_Revenue_Rate'] = (
    customer_value['AvgOrderValue'] * customer_value['PurchaseFreq_Monthly']
)
customer_value['Annualized_Revenue_RunRate'] = customer_value['Monthly_Revenue_Rate'] * 12


In [ ]:
# Sanity check: historical revenue should equal average order value × order frequency.
customer_value['Revenue_Check'] = (
    customer_value['AvgOrderValue'] * customer_value['Frequency']
)
print(f"Maximum absolute revenue difference: {(customer_value['TotalRevenue'] - customer_value['Revenue_Check']).abs().max():.10f}")


### 4.3 Merging Customer Value Metrics into the Main Segmentation Table


In [ ]:
rfm = rfm.merge(
    customer_value[['CustomerID', 'TotalRevenue', 'Annualized_Revenue_RunRate']],
    on='CustomerID'
)


### 4.4 Average Annualized Revenue Run-Rate by Segment


In [ ]:
segment_run_rate = rfm.groupby('Segment')['Annualized_Revenue_RunRate'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=segment_run_rate.values, y=segment_run_rate.index,
            hue=segment_run_rate.index, legend=False, palette='crest')
plt.title('Average Annualized Revenue Run-Rate by Segment')
plt.xlabel('Average Annualized Revenue (£)')
plt.ylabel('Segment')
plt.tight_layout()
plt.show()


In [ ]:
segment_summary = pd.DataFrame({
    'Customer_Count': rfm['Segment'].value_counts(),
    'Total_Revenue': segment_revenue,
    'Revenue_Pct': segment_pct,
    'Avg_Annualized_Revenue': rfm.groupby('Segment')['Annualized_Revenue_RunRate'].mean().round(2)
}).sort_values('Total_Revenue', ascending=False)

print(segment_summary)


## 5. Key Findings

### 1. Champions are a small but highly valuable customer group
Champions account for about **25% of customers but approximately 69% of revenue**, showing strong concentration of business value in a relatively small segment.

**Implication:** Champions should be protected through loyalty initiatives, personalized engagement, and cross-sell/upsell opportunities.

### 2. Customer count alone does not indicate business value
Lost customers are one of the largest segments by customer count, but they contribute only about **4% of total revenue**. Champions have a similar-sized customer base but contribute far more revenue.

**Implication:** retention resources should be allocated using customer value and behavior rather than segment size alone.

### 3. At Risk customers are the highest-priority retention target, not Lost
At Risk customers have a substantially higher **annualized historical revenue run-rate** than Lost customers. This indicates that At Risk customers had stronger purchasing intensity before becoming inactive.

Because this segment is relatively small, a targeted win-back campaign can focus resources on customers with stronger historical value rather than applying the same intervention to the much larger Lost segment.

**Recommendation:** prioritize retention/win-back spend on **At Risk** over **Lost**. Lost customers can be addressed through lower-cost, low-touch re-engagement, while At Risk customers justify more personalized outreach.

**Important limitation:** the annualized revenue run-rate assumes the observed purchase rate continues for the next 12 months. It does not model future churn, retention probability, discounting, or margin, so it should not be interpreted as predictive CLV.


### 3. At Risk customers are the highest-priority retention target, not Lost
This is the central insight of the analysis. At Risk customers have an 
average annualized revenue run-rate of **£4,762.54**, about 84% of Champions' 
**£5,650.62**, and more than 3× Lost's **£1,565.05**. This indicates that At Risk 
customers had substantially higher historical purchasing intensity before becoming inactive.

Because this segment is also relatively small (353 customers, about 6% of the 
base), a targeted win-back campaign can focus resources on customers with stronger 
historical value rather than applying the same intervention to the much larger Lost segment.

**Recommendation:** prioritize retention/win-back spend on **At Risk** over **Lost**. 
Lost customers can be addressed through lower-cost, low-touch re-engagement, while 
At Risk customers justify more personalized outreach given their stronger historical revenue rate.

**Important limitation:** the annualized revenue run-rate assumes the observed purchase 
rate continues for the next 12 months. It does not model future churn, retention probability, 
discounting, or margin, so it should not be interpreted as predictive CLV.

This analysis combines RFM segmentation with customer value analysis to prioritize retention effort. The headline result is that Champions represent about 25% of customers but generate about 69% of revenue, while At Risk customers show a substantially higher historical revenue run-rate than Lost customers.

The key practical recommendation is to prioritize targeted win-back activity for **At Risk customers** over broad investment in the much larger Lost segment. Lost customers can still be addressed through lower-cost, low-touch re-engagement.

**Limitations:** the annualized revenue run-rate is a descriptive projection based on observed purchasing behavior. It does not predict whether customers will remain active, and it does not account for profit margin or discounting.

**Potential extensions:**
- Build a predictive CLV model such as BG/NBD + Gamma-Gamma.
- Build cohort retention curves to understand how churn risk changes over customer tenure.
- Test the At-Risk-over-Lost prioritization with an A/B win-back campaign.
- Segment by product category or country to test whether the same patterns hold across customer subgroups.
